# 2D Pegasus Phase Diagram

Publication-style 2D phase diagrams for RAU and CBFM, plus thermodynamic quantity traces for selected beta1 values.


In [ ]:
from collections import defaultdict
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.collections import PolyCollection
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
from matplotlib.offsetbox import AnnotationBbox, OffsetImage
from matplotlib.patches import Patch
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from scipy.ndimage import gaussian_filter
from scipy.spatial import Voronoi, cKDTree

try:
    from pdf2image import convert_from_path
except ImportError:
    convert_from_path = None


REPO_ROOT = Path.cwd().parent if Path.cwd().name == "plots" else Path.cwd()
PLOTS_DIR = REPO_ROOT / "plots"
TIKZ_DIR = PLOTS_DIR / "tikz"

GRAPH_SIZE = 6
ANNEALING_TIME = 100
MIN_BETA = 0.0
MAX_BETA = 2.0
PHASE_DIAGRAM_BETA_LIMITS = (0.0, 1.0)
MIN_AP = 0.1
MAX_AP = 0.7

DATASET_CONFIGS = {
    "RAU": {
        "title": "RAU",
        "data_path": REPO_ROOT / "data/results" / f"phase_diagram_P{GRAPH_SIZE}_RAU_advantage6.4",
    },
    "CBFM": {
        "title": "CBFM",
        "data_path": REPO_ROOT / "data/results" / f"phase_diagram_P{GRAPH_SIZE}_CBFM_advantage6.4",
    },
}
DATASET_ORDER = ("RAU", "CBFM")

PHASE_DEFINITIONS = {
    0: {"label": "Undetermined", "short_label": "U", "color": "lightgray", "tikz_files": {}, "show_in_legend": False},
    1: {"label": "Refrigerator", "short_label": "R", "color": "blue", "tikz_files": {"beta1_lt_beta2": "refrigerator_beta1_lt_beta2.pdf", "beta1_gt_beta2": "refrigerator_beta1_gt_beta2.pdf"}, "show_in_legend": True},
    2: {"label": "Engine / Accelerator", "short_label": "E", "color": "red", "tikz_files": {"beta1_lt_beta2": "engine_beta1_lt_beta2.pdf", "beta1_gt_beta2": "engine_beta1_gt_beta2.pdf"}, "show_in_legend": True},
    3: {"label": "Accelerator", "short_label": "A", "color": "green", "tikz_files": {"beta1_lt_beta2": "accelerator_beta1_lt_beta2.pdf", "beta1_gt_beta2": "accelerator_beta1_gt_beta2.pdf"}, "show_in_legend": True},
    4: {"label": "Heater", "short_label": "H", "color": "orange", "tikz_files": {"beta1_lt_beta2": "heater_beta1_lt_beta2.pdf", "beta1_gt_beta2": "heater_beta1_gt_beta2.pdf"}, "show_in_legend": True},
}
LEGEND_PHASE_IDS = (1, 2, 3, 4)
PHASE_LABELS = {phase: definition["label"] for phase, definition in PHASE_DEFINITIONS.items()}
PHASE_COLORS = {phase: definition["color"] for phase, definition in PHASE_DEFINITIONS.items()}
PHASE_TIKZ_FILES = {phase: definition["tikz_files"] for phase, definition in PHASE_DEFINITIONS.items() if definition["tikz_files"]}
BETA_ORDER_LABELS = {
    "beta1_lt_beta2": r"$\beta_1 < \beta_2$",
    "beta1_gt_beta2": r"$\beta_1 > \beta_2$",
}

SELECTED_BETA1_VALUES = (0.2, 0.7, 0.2, 0.5, 1.0)
SELECTED_BETA1_VALUES_BY_DATASET = {
    "RAU": (0.1, 0.5, 0.85),
    "CBFM": (0.1, 0.3, 0.45),
}
BETA1_LINESTYLES = ("-", "--", ":", "-.", (0, (3, 1, 1, 1)))

INEQUALITY_TOL = 1e-3
K = 2
CLEAN_ISOLATED_PHASE_POINTS = True
DRAW_CELL_BOUNDARY = False
DRAW_SAMPLE_POINTS = True
DRAW_VORONOI_PHASE_CELLS = False
SMOOTH_PHASE_COLORS = True
SMOOTH_PHASE_COLOR_GRID_SIZE = 900
SMOOTH_PHASE_COLOR_SIGMA = 18.0
SMOOTH_PHASE_COLOR_ALPHA = 0.72
SHOW_FUZZY_BOUNDARY = False
DRAW_INTERPOLATED_PHASE_BOUNDARY = True
FUZZY_BOUNDARY_COLOR = "black"
FUZZY_BOUNDARY_WIDTHS = (1.0, 0.75, 0.2)
FUZZY_BOUNDARY_ALPHAS = (0.08, 0.18, 0.1)
FUZZY_BOUNDARY_GRID_SIZE = 1000
FUZZY_BOUNDARY_SMOOTHING = 1.0
UNCERTAIN_BOUNDARY_EXCLUDED_REGIONS = []
INTERPOLATED_PHASE_BOUNDARY_GRID_SIZE = 900
INTERPOLATED_PHASE_BOUNDARY_SMOOTHING = 25.0
INTERPOLATED_PHASE_BOUNDARY_DEFAULT_STYLE = {"color": "black", "linewidth": 1.8, "linestyle": "-", "alpha": 1.0}
INTERPOLATED_PHASE_BOUNDARY_MIN_LOCAL_MEMBERSHIP = 0.08
INTERPOLATED_PHASE_BOUNDARY_STYLES = [
    {"phase_pair": (1, 3), "color": "black", "linewidth": 5.0, "linestyle": "-", "alpha": 1.0},
    {"phase_pair": (2, 3), "color": "black", "linewidth": 5.0, "linestyle": "--", "alpha": 1.0},
]

BETA_COMPARISON_LABELS = {
    0: r"$\beta_1 \approx \beta_2$",
    1: r"$\beta_1 < \beta_2$",
    2: r"$\beta_1 > \beta_2$",
}

BETA_COMPARISON_COLORS = {
    0: "0.65",
    1: "lightblue",
    2: "lightcoral",
}

B0 = 1.0
H_PLANCK = 6.62607015e-34
K_BOLTZMANN = 1.380649e-23
UNITS_FACTOR = (B0 / 2.0) * 10**9 * H_PLANCK
BETA_UNIT = K_BOLTZMANN / UNITS_FACTOR


def latex_plot(scale: float = 1.0, fontsize: int = 12) -> None:
    """Set publication-style matplotlib defaults."""
    fig_width_pt = 246.0
    inches_per_pt = 1.0 / 72.27
    golden_mean = (np.sqrt(5.0) - 1.0) / 2.0
    fig_width = fig_width_pt * inches_per_pt * scale
    fig_height = fig_width * golden_mean
    mpl.rcParams.update(
        {
            "pgf.texsystem": "pdflatex",
            "text.usetex": True,
            "font.family": "serif",
            "axes.labelsize": fontsize,
            "font.size": fontsize,
            "legend.fontsize": fontsize,
            "xtick.labelsize": 12,
            "ytick.labelsize": 12,
            "figure.figsize": [fig_width, fig_height],
        }
    )


In [ ]:
def as_dict(obj):
    if isinstance(obj, dict):
        return obj
    if hasattr(obj, "item"):
        maybe = obj.item()
        if isinstance(maybe, dict):
            return maybe
    raise TypeError(f"Expected a dict-like object, got {type(obj)}")


def average_over_instances(beta_raw, q_raw, energies_raw=None):
    beta_group = defaultdict(list)
    q_mean_group = defaultdict(list)
    q_var_group = defaultdict(list)
    energy_mean_group = defaultdict(list)
    energy_var_group = defaultdict(list)

    for key, beta2 in beta_raw.items():
        _, beta1, annealing_time, ap = key
        beta_group[(beta1, annealing_time, ap)].append(float(beta2))

    for key, (q_mean, q_var) in q_raw.items():
        _, beta1, annealing_time, ap = key
        q_mean_group[(beta1, annealing_time, ap)].append(float(q_mean))
        q_var_group[(beta1, annealing_time, ap)].append(float(q_var))

    if energies_raw is not None:
        for key, (e_mean, e_var) in energies_raw.items():
            _, beta1, annealing_time, ap = key
            energy_mean_group[(beta1, annealing_time, ap)].append(float(e_mean))
            energy_var_group[(beta1, annealing_time, ap)].append(float(e_var))

    beta_avg = {
        ("avg", beta1, annealing_time, ap): float(np.mean(values))
        for (beta1, annealing_time, ap), values in beta_group.items()
    }
    q_avg = {
        ("avg", beta1, annealing_time, ap): (
            float(np.mean(q_mean_group[(beta1, annealing_time, ap)])),
            float(np.mean(q_var_group[(beta1, annealing_time, ap)])),
        )
        for beta1, annealing_time, ap in q_mean_group
    }

    if energies_raw is None:
        return beta_avg, q_avg, None

    energies_avg = {
        ("avg", beta1, annealing_time, ap): (
            float(np.mean(energy_mean_group[(beta1, annealing_time, ap)])),
            float(np.mean(energy_var_group[(beta1, annealing_time, ap)])),
        )
        for beta1, annealing_time, ap in energy_mean_group
    }
    return beta_avg, q_avg, energies_avg


def load_dataset(dataset_name):
    config = DATASET_CONFIGS[dataset_name]
    data_path = config["data_path"]
    beta_raw = as_dict(np.load(data_path / f"betas2_P{GRAPH_SIZE}.pkl", allow_pickle=True))
    q_raw = as_dict(np.load(data_path / f"Q_P{GRAPH_SIZE}.pkl", allow_pickle=True))
    energies_path = data_path / f"energies_P{GRAPH_SIZE}.pkl"
    energies_raw = as_dict(np.load(energies_path, allow_pickle=True)) if energies_path.exists() else None
    return average_over_instances(beta_raw, q_raw, energies_raw)


def g_func(x):
    x = np.clip(x, -0.999999, 0.999999)
    return x * np.arctanh(x)


def classify(beta1, beta2, dE1, dE2, work, tol=INEQUALITY_TOL):
    if min(abs(dE1), abs(dE2), abs(work), abs(beta1 - beta2)) <= tol:
        return 0

    if beta1 < beta2:
        if dE1 > tol and dE2 < -tol and work > tol:
            return 1
        if dE1 < -tol and dE2 > tol and work < -tol:
            return 2
        if dE1 < -tol and dE2 > tol and work > tol:
            return 3
        if dE1 > tol and dE2 > tol and work > tol:
            return 4
    else:
        if dE1 < -tol and dE2 > tol and work > tol:
            return 1
        if dE1 > tol and dE2 < -tol and work < -tol:
            return 2
        if dE1 > tol and dE2 < -tol and work > tol:
            return 3
        if dE1 > tol and dE2 > tol and work > tol:
            return 4

    return 0


def iter_filtered_records(beta_raw, q_raw):
    for key, beta2_raw in beta_raw.items():
        if key not in q_raw:
            continue

        _, beta1, annealing_time, ap = key
        beta1 = float(beta1)
        annealing_time = float(annealing_time)
        ap = float(ap)

        if annealing_time != float(ANNEALING_TIME):
            continue
        if not (MIN_BETA <= beta1 <= MAX_BETA and MIN_AP <= ap <= MAX_AP):
            continue

        q_mean, q_var = q_raw[key]
        q_mean = float(q_mean)
        q_var = float(q_var)
        beta2 = -float(beta2_raw)
        if not np.isfinite(beta2) or abs(beta2) <= 1e-10:
            continue

        denominator = np.sqrt(q_var + q_mean**2)
        if not np.isfinite(denominator) or denominator <= 1e-12:
            continue

        ratio = q_mean / denominator
        dE1 = q_mean
        dE2 = (2 / beta2) * g_func(ratio) - (beta1 / beta2) * q_mean
        work = dE1 + dE2
        phase = classify(beta1, beta2, dE1, dE2, work)
        yield {
            "s": ap,
            "beta1": beta1,
            "beta2": beta2,
            "dE1": dE1,
            "dE2": dE2,
            "W": work,
            "phase": phase,
        }


def build_dataset(dataset_name):
    beta_raw, q_raw, energies_raw = load_dataset(dataset_name)
    records = list(iter_filtered_records(beta_raw, q_raw))
    if not records:
        raise ValueError(f"No data points survived filtering for {dataset_name}.")

    beta_values = np.array(sorted({record["beta1"] for record in records}), dtype=float)
    ap_values = np.array(sorted({record["s"] for record in records}), dtype=float)
    beta_index = {value: idx for idx, value in enumerate(beta_values)}
    ap_index = {value: idx for idx, value in enumerate(ap_values)}
    phase_grid = np.full((len(ap_values), len(beta_values)), np.nan)
    beta2_grid = np.full_like(phase_grid, np.nan, dtype=float)

    for record in records:
        row = ap_index[record["s"]]
        col = beta_index[record["beta1"]]
        phase_grid[row, col] = record["phase"]
        beta2_grid[row, col] = record["beta2"]

    if CLEAN_ISOLATED_PHASE_POINTS:
        phase_grid, replacements = clean_isolated_points(phase_grid, beta_values, ap_values)
    else:
        replacements = []

    return {
        "name": dataset_name,
        "title": DATASET_CONFIGS[dataset_name]["title"],
        "beta_raw": beta_raw,
        "q_raw": q_raw,
        "energies_raw": energies_raw,
        "records": records,
        "beta_values": beta_values,
        "ap_values": ap_values,
        "phase_grid": phase_grid,
        "beta2_grid": beta2_grid,
        "replacements": replacements,
    }


def clean_isolated_points(phases, beta_values, ap_values):
    cleaned = phases.copy()
    replacements = []
    defined_positions = np.argwhere(np.isfinite(phases) & (phases != 0))
    for row in range(phases.shape[0]):
        for col in range(phases.shape[1]):
            phase = phases[row, col]
            if not np.isfinite(phase):
                continue
            if int(phase) == 0:
                if len(defined_positions) == 0:
                    continue
                distances = np.sum((defined_positions - np.array([row, col])) ** 2, axis=1)
                nearest_distance = distances.min()
                nearest_positions = defined_positions[distances == nearest_distance]
                nearest_phases = [int(phases[r, c]) for r, c in nearest_positions]
                new_phase = max(set(nearest_phases), key=lambda candidate: (nearest_phases.count(candidate), -candidate))
                cleaned[row, col] = new_phase
                replacements.append((ap_values[row], beta_values[col], 0, new_phase, 0))
                continue
            neighbors = []
            for drow in (-1, 0, 1):
                for dcol in (-1, 0, 1):
                    if drow == 0 and dcol == 0:
                        continue
                    r = row + drow
                    c = col + dcol
                    if 0 <= r < phases.shape[0] and 0 <= c < phases.shape[1]:
                        neighbor = phases[r, c]
                        if np.isfinite(neighbor) and int(neighbor) != 0:
                            neighbors.append(int(neighbor))
            if not neighbors:
                continue
            phase = int(phase)
            same_count = sum(neighbor == phase for neighbor in neighbors)
            if same_count > K:
                continue
            candidates = {neighbor: neighbors.count(neighbor) for neighbor in set(neighbors) if neighbor != phase}
            if not candidates:
                continue
            new_phase, new_count = max(candidates.items(), key=lambda item: item[1])
            if list(candidates.values()).count(new_count) == 1 and new_count > same_count:
                cleaned[row, col] = new_phase
                replacements.append((ap_values[row], beta_values[col], phase, new_phase, same_count))
    return cleaned, replacements


def print_phase_counts(data):
    values = data["phase_grid"][np.isfinite(data["phase_grid"])].astype(int)
    print(f"{data['name']} phase counts:")
    for phase in sorted(np.unique(values)):
        print(f"  {PHASE_LABELS[phase]}: {np.sum(values == phase)}")


In [ ]:
def voronoi_finite_polygons_2d(vor, radius=None):
    if vor.points.shape[1] != 2:
        raise ValueError("Voronoi input must be 2D")
    new_regions = []
    new_vertices = vor.vertices.tolist()
    center = vor.points.mean(axis=0)
    if radius is None:
        radius = np.ptp(vor.points, axis=0).max() * 2
    all_ridges = {}
    for (p1, p2), (v1, v2) in zip(vor.ridge_points, vor.ridge_vertices):
        all_ridges.setdefault(p1, []).append((p2, v1, v2))
        all_ridges.setdefault(p2, []).append((p1, v1, v2))
    for p1, region_idx in enumerate(vor.point_region):
        vertices = vor.regions[region_idx]
        if all(v >= 0 for v in vertices):
            new_regions.append(vertices)
            continue
        new_region = [v for v in vertices if v >= 0]
        for p2, v1, v2 in all_ridges[p1]:
            if v2 < 0:
                v1, v2 = v2, v1
            if v1 >= 0:
                continue
            tangent = vor.points[p2] - vor.points[p1]
            tangent /= np.linalg.norm(tangent)
            normal = np.array([-tangent[1], tangent[0]])
            midpoint = vor.points[[p1, p2]].mean(axis=0)
            direction = np.sign(np.dot(midpoint - center, normal)) * normal
            far_point = vor.vertices[v2] + direction * radius
            new_vertices.append(far_point.tolist())
            new_region.append(len(new_vertices) - 1)
        vertices_array = np.asarray([new_vertices[v] for v in new_region])
        angles = np.arctan2(vertices_array[:, 1] - vertices_array[:, 1].mean(), vertices_array[:, 0] - vertices_array[:, 0].mean())
        new_regions.append([v for _, v in sorted(zip(angles, new_region))])
    return new_regions, np.asarray(new_vertices)


def clip_polygon_to_box(polygon, xmin, xmax, ymin, ymax):
    def clip_edge(points, inside, intersect):
        if len(points) == 0:
            return points
        output = []
        prev = points[-1]
        prev_inside = inside(prev)
        for current in points:
            current_inside = inside(current)
            if current_inside:
                if not prev_inside:
                    output.append(intersect(prev, current))
                output.append(current)
            elif prev_inside:
                output.append(intersect(prev, current))
            prev = current
            prev_inside = current_inside
        return np.array(output)

    polygon = np.asarray(polygon, dtype=float)
    if len(polygon) == 0:
        return polygon
    eps = 1e-12
    polygon = clip_edge(polygon, lambda p: p[0] >= xmin, lambda p1, p2: p1 + (p2 - p1) * ((xmin - p1[0]) / (p2[0] - p1[0] + eps)))
    polygon = clip_edge(polygon, lambda p: p[0] <= xmax, lambda p1, p2: p1 + (p2 - p1) * ((xmax - p1[0]) / (p2[0] - p1[0] + eps)))
    polygon = clip_edge(polygon, lambda p: p[1] >= ymin, lambda p1, p2: p1 + (p2 - p1) * ((ymin - p1[1]) / (p2[1] - p1[1] + eps)))
    polygon = clip_edge(polygon, lambda p: p[1] <= ymax, lambda p1, p2: p1 + (p2 - p1) * ((ymax - p1[1]) / (p2[1] - p1[1] + eps)))
    return polygon


def points_in_rectangles(x, y, rectangles):
    mask = np.zeros(np.shape(x), dtype=bool)
    for rectangle in rectangles:
        x_min, x_max = rectangle["x"]
        y_min, y_max = rectangle["y"]
        mask |= (x_min <= x) & (x <= x_max) & (y_min <= y) & (y <= y_max)
    return mask


def build_voronoi_polygons(points, values, x_limits, y_limits):
    if len(points) < 4:
        raise ValueError("At least 4 points are needed to draw Voronoi cells.")
    vor = Voronoi(points)
    regions, vertices = voronoi_finite_polygons_2d(vor)
    polygons = []
    polygon_values = []
    for region, value in zip(regions, values):
        polygon = clip_polygon_to_box(vertices[region], x_limits[0], x_limits[1], y_limits[0], y_limits[1])
        if len(polygon) >= 3:
            polygons.append(polygon)
            polygon_values.append(int(value))
    return polygons, np.array(polygon_values, dtype=int), vor


def add_smoothed_phase_colors(ax, points, phases, x_limits, y_limits):
    if not SMOOTH_PHASE_COLORS:
        return False
    present_phases = sorted(set(int(phase) for phase in phases if np.isfinite(phase)))
    if not present_phases:
        return False
    ap_grid = np.linspace(x_limits[0], x_limits[1], SMOOTH_PHASE_COLOR_GRID_SIZE)
    beta_grid = np.linspace(y_limits[0], y_limits[1], SMOOTH_PHASE_COLOR_GRID_SIZE)
    ap_surface, beta_surface = np.meshgrid(ap_grid, beta_grid)
    surface_points = np.column_stack([ap_surface.ravel(), beta_surface.ravel()])
    _, nearest_indices = cKDTree(points).query(surface_points, k=1)
    phase_surface = phases[nearest_indices].reshape(ap_surface.shape).astype(int)
    memberships = []
    colors = []
    for phase in present_phases:
        memberships.append(gaussian_filter((phase_surface == phase).astype(float), sigma=SMOOTH_PHASE_COLOR_SIGMA))
        colors.append(mpl.colors.to_rgb(PHASE_COLORS[phase]))
    memberships = np.stack(memberships, axis=-1)
    membership_sum = memberships.sum(axis=-1, keepdims=True)
    membership_sum[membership_sum <= 1e-12] = 1.0
    weights = memberships / membership_sum
    rgb = np.tensordot(weights, np.asarray(colors, dtype=float), axes=([-1], [0]))
    rgba = np.empty((*rgb.shape[:2], 4), dtype=float)
    rgba[..., :3] = rgb
    rgba[..., 3] = SMOOTH_PHASE_COLOR_ALPHA
    ax.imshow(
        rgba,
        extent=[x_limits[0], x_limits[1], y_limits[0], y_limits[1]],
        origin="lower",
        aspect="auto",
        interpolation="bilinear",
        zorder=1,
    )
    return True


def add_fuzzy_phase_boundary(ax, points, phases, vor):
    if not SHOW_FUZZY_BOUNDARY:
        return False
    uncertain_mask = np.zeros(len(points), dtype=bool)
    for p1, p2 in vor.ridge_points:
        if phases[p1] != phases[p2]:
            uncertain_mask[p1] = True
            uncertain_mask[p2] = True
    uncertain_mask[points_in_rectangles(points[:, 0], points[:, 1], UNCERTAIN_BOUNDARY_EXCLUDED_REGIONS)] = False
    if not np.any(uncertain_mask):
        return False
    ap_grid = np.linspace(MIN_AP, MAX_AP, FUZZY_BOUNDARY_GRID_SIZE)
    beta_grid = np.linspace(MIN_BETA, MAX_BETA, FUZZY_BOUNDARY_GRID_SIZE)
    ap_surface, beta_surface = np.meshgrid(ap_grid, beta_grid)
    surface_points = np.column_stack([ap_surface.ravel(), beta_surface.ravel()])
    _, nearest_indices = cKDTree(points).query(surface_points, k=1)
    uncertain_surface = uncertain_mask[nearest_indices].reshape(ap_surface.shape).astype(float)
    uncertain_surface = gaussian_filter(uncertain_surface, sigma=FUZZY_BOUNDARY_SMOOTHING)
    uncertain_surface[points_in_rectangles(ap_surface, beta_surface, UNCERTAIN_BOUNDARY_EXCLUDED_REGIONS)] = np.nan
    for fuzzy_width, fuzzy_alpha in zip(FUZZY_BOUNDARY_WIDTHS, FUZZY_BOUNDARY_ALPHAS):
        ax.contour(ap_surface, beta_surface, uncertain_surface, levels=[0.5], colors=FUZZY_BOUNDARY_COLOR, linewidths=fuzzy_width, alpha=fuzzy_alpha, zorder=7)
    return True


def add_interpolated_phase_boundaries(ax, points, phases, x_limits, y_limits):
    if not DRAW_INTERPOLATED_PHASE_BOUNDARY:
        return False
    present_phases = sorted(set(int(phase) for phase in phases if np.isfinite(phase)))
    if len(present_phases) < 2:
        return False
    neighboring_phase_pairs = sorted(
        {
            tuple(sorted((int(phases[p1]), int(phases[p2]))))
            for p1, p2 in Voronoi(points).ridge_points
            if int(phases[p1]) != int(phases[p2]) and 0 not in (int(phases[p1]), int(phases[p2]))
        }
    )
    if not neighboring_phase_pairs:
        return False
    ap_grid = np.linspace(x_limits[0], x_limits[1], INTERPOLATED_PHASE_BOUNDARY_GRID_SIZE)
    beta_grid = np.linspace(y_limits[0], y_limits[1], INTERPOLATED_PHASE_BOUNDARY_GRID_SIZE)
    ap_surface, beta_surface = np.meshgrid(ap_grid, beta_grid)
    surface_points = np.column_stack([ap_surface.ravel(), beta_surface.ravel()])
    _, nearest_indices = cKDTree(points).query(surface_points, k=1)
    phase_surface = phases[nearest_indices].reshape(ap_surface.shape).astype(int)
    style_by_pair = {
        tuple(sorted(style["phase_pair"])): style
        for style in INTERPOLATED_PHASE_BOUNDARY_STYLES
        if "phase_pair" in style
    }
    fallback_styles = [style for style in INTERPOLATED_PHASE_BOUNDARY_STYLES if "phase_pair" not in style]
    detected_boundary_count = 0
    for phase_pair in neighboring_phase_pairs:
        phase_a, phase_b = phase_pair
        membership_a = gaussian_filter((phase_surface == phase_a).astype(float), sigma=INTERPOLATED_PHASE_BOUNDARY_SMOOTHING)
        membership_b = gaussian_filter((phase_surface == phase_b).astype(float), sigma=INTERPOLATED_PHASE_BOUNDARY_SMOOTHING)
        local_membership = membership_a + membership_b
        pair_surface = np.ma.masked_where(local_membership < INTERPOLATED_PHASE_BOUNDARY_MIN_LOCAL_MEMBERSHIP, membership_a - membership_b)
        finite_values = pair_surface.compressed()
        if len(finite_values) == 0 or finite_values.min() > 0 or finite_values.max() < 0:
            continue
        probe = ax.contour(ap_surface, beta_surface, pair_surface, levels=[0.0], colors="black", linewidths=0.0, alpha=0.0)
        if hasattr(probe, "get_paths"):
            has_boundary = any(len(path.vertices) > 0 for path in probe.get_paths())
        else:
            has_boundary = any(len(segment) > 0 for level_segments in probe.allsegs for segment in level_segments)
        if hasattr(probe, "remove"):
            probe.remove()
        elif hasattr(probe, "collections"):
            for collection in probe.collections:
                collection.remove()
        if not has_boundary:
            continue
        style = INTERPOLATED_PHASE_BOUNDARY_DEFAULT_STYLE.copy()
        if phase_pair in style_by_pair:
            style.update(style_by_pair[phase_pair])
        elif detected_boundary_count < len(fallback_styles):
            style.update(fallback_styles[detected_boundary_count])
        color = style.get("color", "black")
        linewidth = style.get("linewidth", 1.8)
        linestyle = style.get("linestyle", "-")
        alpha = style.get("alpha", 1.0)
        zorder = style.get("zorder", 8 + detected_boundary_count)
        ax.contour(
            ap_surface,
            beta_surface,
            pair_surface,
            levels=[0.0],
            colors=color,
            linewidths=linewidth,
            linestyles=linestyle,
            alpha=alpha,
            zorder=zorder,
        )
        detected_boundary_count += 1
    return detected_boundary_count > 0


def pdf_to_image_with_background(pdf_path, dpi=300, bg_alpha=0.9):
    if convert_from_path is None:
        raise ImportError("pdf2image is required to render TikZ PDF insets.")
    pages = convert_from_path(str(pdf_path), dpi=dpi)
    img = pages[0].convert("RGBA")
    img_array = np.array(img)
    white_threshold = 250
    white_mask = (img_array[:, :, 0] > white_threshold) & (img_array[:, :, 1] > white_threshold) & (img_array[:, :, 2] > white_threshold)
    img_array[white_mask, 3] = int(bg_alpha * 255)
    return img_array


def representative_phase_positions(points, phases):
    positions = {}
    for phase in sorted(set(phases)):
        if phase == 0:
            continue
        phase_points = points[phases == phase]
        if len(phase_points) == 0:
            continue
        center = np.median(phase_points, axis=0)
        nearest = phase_points[np.argmin(np.linalg.norm(phase_points - center, axis=1))]
        positions[int(phase)] = (float(nearest[0]), float(nearest[1]))
    return positions


def beta_order_from_record(record, tol=INEQUALITY_TOL):
    if record["beta1"] < record["beta2"] - tol:
        return "beta1_lt_beta2"
    if record["beta1"] > record["beta2"] + tol:
        return "beta1_gt_beta2"
    return None


def beta_order_at_position(data, position):
    record_points = np.array([(record["s"], record["beta1"]) for record in data["records"]], dtype=float)
    if len(record_points) == 0:
        return None
    _, nearest_index = cKDTree(record_points).query(np.asarray(position, dtype=float), k=1)
    return beta_order_from_record(data["records"][int(nearest_index)])


def phase_tikz_file_for_order(phase, beta_order):
    tikz_files = PHASE_TIKZ_FILES.get(phase, {})
    if beta_order is None:
        return None
    return tikz_files.get(beta_order)


def add_mode_insets(ax, data, points, phases, positions=None, zoom=0.075, bg_alpha=0.92):
    if not positions:
        return []
    manual_positions = {}
    for phase, value in positions.items():
        if value is None:
            continue
        if isinstance(value, dict):
            manual_positions[int(phase)] = value
        else:
            value = tuple(value)
            manual_positions[int(phase)] = {"position": value[:2], "zoom": value[2] if len(value) > 2 else zoom}
    positions = manual_positions
    present_phases = set(int(phase) for phase in phases)
    added = []
    for phase, position in positions.items():
        if phase not in present_phases:
            continue
        if isinstance(position, dict):
            inset_position = position["position"]
            inset_zoom = position.get("zoom", zoom)
        else:
            inset_position = position
            inset_zoom = zoom
        beta_order = beta_order_at_position(data, inset_position)
        pdf_name = phase_tikz_file_for_order(phase, beta_order)
        if pdf_name is None:
            print(f"Skipping {PHASE_LABELS[phase]} inset; no cartoon configured for beta order {beta_order}")
            continue
        pdf_path = TIKZ_DIR / pdf_name
        if not pdf_path.exists():
            print(f"Skipping {PHASE_LABELS[phase]} inset; missing {pdf_path}")
            continue
        try:
            image = pdf_to_image_with_background(pdf_path, dpi=300, bg_alpha=bg_alpha)
        except Exception as exc:
            print(f"Skipping {PHASE_LABELS[phase]} inset; could not load {pdf_path}: {exc}")
            continue
        imagebox = OffsetImage(image, zoom=inset_zoom)
        inset = AnnotationBbox(imagebox, inset_position, frameon=True, box_alignment=(0.5, 0.5), bboxprops={"boxstyle": "round,pad=0.12", "facecolor": "white", "edgecolor": "0.25", "alpha": 0.88}, zorder=10)
        ax.add_artist(inset)
        added.append(f"{PHASE_LABELS[phase]} ({BETA_ORDER_LABELS[beta_order]})")
    return added


In [ ]:
datasets = {name: build_dataset(name) for name in DATASET_ORDER}

for name, data in datasets.items():
    print(f"{name}: loaded {len(data['records'])} filtered records")
    print(f"  grid shape: {data['phase_grid'].shape[0]} s values x {data['phase_grid'].shape[1]} beta1 values")
    print(f"  beta1 range: [{data['beta_values'].min():g}, {data['beta_values'].max():g}]")
    print(f"  s range: [{data['ap_values'].min():g}, {data['ap_values'].max():g}]")
    if data["replacements"]:
        print(f"  isolated-point replacements: {len(data['replacements'])}")
    print_phase_counts(data)


In [ ]:
def add_beta_comparison_inset(ax, data, x_limits, y_limits):
    ax_inset = inset_axes(ax, width="25%", height="25%", loc="lower left", borderpad=1.5, bbox_to_anchor=(0.02, 0.02, 0.96, 0.96), bbox_transform=ax.transAxes)
    beta2_grid = data["beta2_grid"]
    beta_mesh, ap_mesh = np.meshgrid(data["beta_values"], data["ap_values"])
    finite = np.isfinite(beta2_grid)
    comparison_grid = np.full(beta2_grid.shape, 0, dtype=int)
    comparison_grid[finite & (beta_mesh < beta2_grid - INEQUALITY_TOL)] = 1
    comparison_grid[finite & (beta_mesh > beta2_grid + INEQUALITY_TOL)] = 2
    points = np.column_stack([ap_mesh[finite], beta_mesh[finite]])
    comparison_values = comparison_grid[finite].astype(int)
    if len(points) >= 4:
        comparison_polygons, polygon_values, _ = build_voronoi_polygons(points, comparison_values, x_limits=x_limits, y_limits=y_limits)
        cells = PolyCollection(
            comparison_polygons,
            facecolors=[BETA_COMPARISON_COLORS[int(value)] for value in polygon_values],
            edgecolors="none",
            linewidths=0.0,
            alpha=0.88,
            zorder=1,
        )
        ax_inset.add_collection(cells)
    elif len(points):
        ax_inset.scatter(points[:, 0], points[:, 1], c=[BETA_COMPARISON_COLORS[int(value)] for value in comparison_values], s=5, linewidths=0)
    # ax_inset.set_xlabel(r"$s$", fontsize=7)
    # ax_inset.set_ylabel(r"$\beta_1$", fontsize=7)
    ax_inset.tick_params(axis="both", labelsize=10, length=2)
    ax_inset.set_xlim(x_limits)
    ax_inset.set_ylim(y_limits)
    ax_inset.set_facecolor("white")
    ax_inset.legend(
        handles=[
            Patch(facecolor=BETA_COMPARISON_COLORS[1], edgecolor="black", label=BETA_COMPARISON_LABELS[1]),
            Patch(facecolor=BETA_COMPARISON_COLORS[2], edgecolor="black", label=BETA_COMPARISON_LABELS[2]),
        ],
        loc="upper left",
        fontsize=6,
        framealpha=0.92,
        borderpad=0.5,
        handlelength=0.8,
    )
    return comparison_grid


MODE_INSET_POSITION_LIST = [
    ("CBFM", 1, 0.62, 0.82, 0.13),
    ("RAU", 2, 0.39, 0.2, 0.13),
    ("RAU", 3, 0.2, 0.8, 0.13),
    # ("CBFM", 1, 0.22, 0.82, 0.065),
    # ("CBFM", 2, 0.52, 0.50, 0.065),
    # ("CBFM", 3, 0.78, 0.22, 0.065),
]
MODE_INSET_POSITIONS = {name: {} for name in DATASET_ORDER}
for dataset_name, phase, s_position, beta1_position, zoom in MODE_INSET_POSITION_LIST:
    MODE_INSET_POSITIONS.setdefault(dataset_name, {})[int(phase)] = (s_position, beta1_position, zoom)

lbs = ['(b)', '(a)']
def plot_phase_panel(ax, data, mode_inset_positions=None):
    x_limits = (MIN_AP, MAX_AP)
    y_limits = PHASE_DIAGRAM_BETA_LIMITS
    beta_mesh, ap_mesh = np.meshgrid(data["beta_values"], data["ap_values"])
    valid = np.isfinite(data["phase_grid"])
    points = np.column_stack([ap_mesh[valid], beta_mesh[valid]])
    phases = data["phase_grid"][valid].astype(int)
    phase_polygons, polygon_phases, vor = build_voronoi_polygons(points, phases, x_limits=x_limits, y_limits=y_limits)

    smoothed_colors_added = add_smoothed_phase_colors(ax, points, phases, x_limits, y_limits)
    if DRAW_VORONOI_PHASE_CELLS or not smoothed_colors_added:
        cells = PolyCollection(
            phase_polygons,
            facecolors=[PHASE_COLORS[phase] for phase in polygon_phases],
            edgecolors="black" if DRAW_CELL_BOUNDARY else "none",
            linewidths=0.25 if DRAW_CELL_BOUNDARY else 0.0,
            alpha=0.72,
            zorder=2,
        )
        ax.add_collection(cells)

    fuzzy_added = add_fuzzy_phase_boundary(ax, points, phases, vor)
    if DRAW_SAMPLE_POINTS:
        ax.scatter(points[:, 0], points[:, 1], s=6, c="black", alpha=0.24, linewidths=0, zorder=4)
    interpolated_boundary_added = add_interpolated_phase_boundaries(ax, points, phases, x_limits, y_limits)
    add_beta_comparison_inset(ax, data, x_limits, y_limits)
    added_insets = add_mode_insets(ax, data, points, phases, positions=mode_inset_positions, zoom=0.065)

    ax.set_xlim(x_limits)
    ax.set_ylim(y_limits)
    ax.set_xlabel(r"Annealing parameter $s$", fontsize=12)
    if ax.get_subplotspec().is_first_col():
        ax.set_ylabel(r"Initial state inverse temperature $\beta_1$", fontsize=12)
    # ax.set_title(f"{data['title']}, P{GRAPH_SIZE}, t={ANNEALING_TIME:g} us")
    ax.text(0.81, 0.05, rf"{lbs.pop()} {data['title']}, $\tau={ANNEALING_TIME:g} \, \mu s$", transform=ax.transAxes, ha="center", va="bottom", fontsize=12)
    ax.grid(True, alpha=0.12)
    return sorted(set(phases)), fuzzy_added, interpolated_boundary_added, added_insets


latex_plot(scale=2.6, fontsize=12)
fig = plt.figure(figsize=(11.0, 5.2), constrained_layout=True)
gs = fig.add_gridspec(1, 2, wspace=0.04)
axes = np.empty(2, dtype=object)
axes[0] = fig.add_subplot(gs[0, 0])
axes[1] = fig.add_subplot(gs[0, 1], sharex=axes[0], sharey=axes[0])
axes[1].tick_params(labelleft=False)

all_present_phases = set()
inset_summary = {}
for ax, name in zip(axes, DATASET_ORDER):
    present_phases, fuzzy_added, interpolated_boundary_added, added_insets = plot_phase_panel(ax, datasets[name], mode_inset_positions=MODE_INSET_POSITIONS.get(name))
    all_present_phases.update(present_phases)
    inset_summary[name] = added_insets

handles = [
    Patch(facecolor=PHASE_COLORS[phase], label=PHASE_LABELS[phase])
    for phase in LEGEND_PHASE_IDS
    if phase in all_present_phases
]
axes[1].legend(handles=handles, loc="upper left", ncol=1, framealpha=0.94)

phase_output_path = PLOTS_DIR / f"phase_diagram_2d_P{GRAPH_SIZE}_RAU_CBFM_at_{ANNEALING_TIME:g}.pdf"
fig.savefig(phase_output_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved {phase_output_path}")
for name, added_insets in inset_summary.items():
    print(f"{name} TikZ insets:", ", ".join(added_insets) if added_insets else "none")


In [ ]:
QUANTITY_SPECS = (
    ("beta2", r"Temperature $T_2$ [mK]"),
    ("dE1", r"QPU energy change $\langle \Delta E_1 \rangle$ [J]"),
    ("dE2", r"Dumped heat bound $\langle Q \rangle$ [J]"),
    ("W", r"Work bound $\langle W \rangle$ [J]"),
)
PANEL_LABELS = tuple("abcdefghijklmnopqrstuvwxyz")


def nearest_available_beta1(beta_values, requested_beta1):
    beta_values = np.asarray(beta_values, dtype=float)
    return float(beta_values[np.argmin(np.abs(beta_values - requested_beta1))])


def selected_beta1_values_for_dataset(dataset_name):
    return SELECTED_BETA1_VALUES_BY_DATASET.get(dataset_name, SELECTED_BETA1_VALUES)


def build_quantity_series(data, requested_beta1_values=None):
    if requested_beta1_values is None:
        requested_beta1_values = selected_beta1_values_for_dataset(data["name"])
    snapped_values = []
    for requested_beta1 in requested_beta1_values:
        snapped = nearest_available_beta1(data["beta_values"], requested_beta1)
        if not any(np.isclose(snapped, existing) for existing in snapped_values):
            snapped_values.append(snapped)

    series_by_beta1 = {}
    for beta1_value in snapped_values:
        rows = [record for record in data["records"] if np.isclose(record["beta1"], beta1_value)]
        rows = sorted(rows, key=lambda record: record["s"])
        if rows:
            series_by_beta1[beta1_value] = rows
    return series_by_beta1


def beta_to_temperature_mk(beta):
    beta = np.asarray(beta, dtype=float)
    return 1000.0 / (beta * BETA_UNIT)


def quantity_plot_values(rows, quantity_key):
    values = np.array([record[quantity_key] for record in rows], dtype=float)
    if quantity_key == "beta2":
        return beta_to_temperature_mk(values)
    if quantity_key == "dE2":
        return -values * UNITS_FACTOR
    return values * UNITS_FACTOR


def plot_quantity_panel(ax, rows, quantity_key, linestyle, label):
    s_values = [record["s"] for record in rows]
    values = quantity_plot_values(rows, quantity_key)
    phases = np.array([int(record["phase"]) for record in rows], dtype=int)
    ax.plot(s_values, values, color="black", linestyle=linestyle, linewidth=1.35, alpha=0.85, label=label)
    for phase in sorted(set(phases)):
        phase_mask = phases == phase
        ax.scatter(
            np.array(s_values)[phase_mask],
            values[phase_mask],
            s=22,
            color=PHASE_COLORS[phase],
            edgecolors="black",
            linewidth=0.3,
            zorder=3,
        )


def beta_legend_label(dataset_name, beta1_value):
    return rf"$\beta_1={beta1_value:g}$"


latex_plot(scale=2.8, fontsize=11)
fig = plt.figure(figsize=(15, 6.6), constrained_layout=True)
gs = fig.add_gridspec(2, 4, wspace=0.1, hspace=0.1)
axes = np.empty((2, 4), dtype=object)
for row_idx in range(2):
    for col_idx in range(4):
        if row_idx == 0 and col_idx == 0:
            axes[row_idx, col_idx] = fig.add_subplot(gs[row_idx, col_idx])
        else:
            axes[row_idx, col_idx] = fig.add_subplot(gs[row_idx, col_idx], sharex=axes[0, 0])
beta_legend_entries_by_dataset = {name: [] for name in DATASET_ORDER}
phase_legend_entries = []

for row_idx, name in enumerate(DATASET_ORDER):
    data = datasets[name]
    series_by_beta1 = build_quantity_series(data)
    for col_idx, (quantity_key, ylabel) in enumerate(QUANTITY_SPECS):
        ax = axes[row_idx, col_idx]
        for line_idx, (beta1_value, rows) in enumerate(series_by_beta1.items()):
            linestyle = BETA1_LINESTYLES[line_idx % len(BETA1_LINESTYLES)]
            label = beta_legend_label(name, beta1_value)
            plot_quantity_panel(ax, rows, quantity_key, linestyle, label)
            if quantity_key == "beta2":
                ax.axhline(beta_to_temperature_mk(beta1_value), color="black", linestyle=linestyle, linewidth=0.65, alpha=1.0, zorder=1)
                if row_idx == 0:
                    ax.set_ylim(20, 80)
                else:
                    ax.set_ylim(40, 100)
            if col_idx == 0:
                beta_legend_entries_by_dataset[name].append(Line2D([0], [0], color="black", linestyle=linestyle, linewidth=1.35, label=label))
        if quantity_key != "beta2":
            ax.axhline(0, color="black", linestyle="--", linewidth=0.9, alpha=0.4)
        panel_idx = row_idx * len(QUANTITY_SPECS) + col_idx
        ax.text(0.7, 0.1, f"({PANEL_LABELS[panel_idx]}) {data['title']}", transform=ax.transAxes, ha="left", va="top", fontsize=11)
        ax.set_ylabel(ylabel)
        if row_idx == len(DATASET_ORDER) - 1:
            ax.set_xlabel(r"Annealing parameter $s$")
        else:
            ax.tick_params(labelbottom=False)
        ax.set_xlim(MIN_AP, MAX_AP)
        ax.grid(True, alpha=0.2)
        if row_idx == 0 and col_idx == 0:
            present_phases = sorted({int(record["phase"]) for rows in series_by_beta1.values() for record in rows})
            phase_legend_entries = [
                Patch(facecolor=PHASE_COLORS[phase], edgecolor="black", label=PHASE_LABELS[phase])
                for phase in LEGEND_PHASE_IDS
                if phase in present_phases
            ]

for row_idx, name in enumerate(DATASET_ORDER):
    entries = list({entry.get_label(): entry for entry in beta_legend_entries_by_dataset[name]}.values())
    beta_legend = axes[row_idx, 0].legend(handles=entries, loc="center left", framealpha=0.94)
    axes[row_idx, 0].add_artist(beta_legend)
axes[0, 0].legend(handles=phase_legend_entries, loc="lower left", bbox_to_anchor=(0.0, 0.7), framealpha=0.94 )

quantity_output_path = PLOTS_DIR / f"thermodynamic_quantities_2d_P{GRAPH_SIZE}_RAU_CBFM_at_{ANNEALING_TIME:g}.pdf"
fig.savefig(quantity_output_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved {quantity_output_path}")
for name in DATASET_ORDER:
    snapped = list(build_quantity_series(datasets[name]).keys())
    print(f"{name} plotted beta1 values:", ", ".join(f"{value:g}" for value in snapped))
